# Solution E3 · Permutation on the training set rewards the noise the forest memorized

**Type**: Find the bug

## Objective

Find out why a report "corrected with the permutation" puts `ticket_code` (a random integer) ahead of `age` and `siblings_spouses`: the permutation was computed on the **training set**, where the forest has memorized the noise too. Redo the computation on the test set and measure the difference.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

df = ds.titanic()
X, y = df.drop(columns="survived").copy(), df["survived"]
r = np.random.default_rng(0)
X["random_number"] = r.uniform(size=len(X))
X["ticket_code"] = r.integers(0, 100000, len(X))
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare", "random_number", "ticket_code"]
CAT = ["sex", "port", "deck", "title"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM), ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), CAT)])
pipe = make_pipeline(prep, RandomForestClassifier(300, random_state=0, n_jobs=-1)).fit(X_tr, y_tr)
NOISE = ["random_number", "ticket_code"]


def importances(A, b, n_repeats=10, seed=0):
    r = permutation_importance(pipe, A, b, n_repeats=n_repeats, random_state=seed, scoring="roc_auc")
    return pd.Series(r.importances_mean, index=A.columns)

## Commented solution

In [ ]:
train_perm = importances(X_tr, y_tr)
train_top5 = list(train_perm.nlargest(5).index)
print("top 5 by permutation (computed on the training set):", train_top5)
print(train_perm.round(3).sort_values(ascending=False).head(7).to_string())

**Where the bug is.** The permutation measures how much the model **gets worse** when a column is shuffled. On the training set the forest has AUC 1.0: it has memorized every row, including the value of the random `ticket_code` (which identifies the passenger). Shuffling that column breaks the identification and the AUC drops,
even though for a new passenger the code says nothing. The same measure on the test set, where memorization does not help, brings it to zero.

In [ ]:
test_perm = importances(X_te, y_te)
test_top5 = list(test_perm.nlargest(5).index)
train_auc = roc_auc_score(y_tr, pipe.predict_proba(X_tr)[:, 1])
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
print("top 5 by permutation (on the test set):", test_top5, f"| training AUC {train_auc:.3f}, test {test_auc:.3f}")

In [ ]:
train_noise = float(train_perm[NOISE].clip(lower=0).sum())
test_noise = float(test_perm[NOISE].clip(lower=0).sum())
train_code_rank = int(train_perm.rank(ascending=False)["ticket_code"])
test_code_rank = int(test_perm.rank(ascending=False)["ticket_code"])
print(f"importance of the two noise columns: training {train_noise:.3f} | test {test_noise:.3f} | position of ticket_code: {train_code_rank} -> {test_code_rank}")

In [ ]:
stable = True
for s in range(3):
    idx = np.random.default_rng(s).choice(len(X_te), len(X_te) // 2, replace=False)
    imp = importances(X_te.iloc[idx], y_te.iloc[idx], n_repeats=5, seed=s)
    stable = stable and bool(imp["ticket_code"] < 0.02)
print("ticket_code stays below 0.02 in all the subsamples of the test set:", stable)

## Check

In [ ]:
with verify():
    check("ticket_code" in train_top5, "the colleague's report puts `ticket_code` in the top five", "don't modify the colleague's code")
    check(train_auc is not None and train_auc > 0.99 and test_auc < 0.9, "training AUC 1.0 (memorization); test below 0.9", "pipe.score(...) is not the AUC: use roc_auc_score with predict_proba")
    check(test_perm is not None and "ticket_code" not in test_top5 and "random_number" not in test_top5 and test_top5[0] == "sex", "on the test set the two noise columns are not in the top five and `sex` is first", "importances(X_te, y_te)")
    check(train_noise > 0.03 and test_noise < 0.02 and train_code_rank < test_code_rank, "the noise is worth over 0.03 on the training set and less than 0.02 on the test set; the ticket code drops in the ranking", "sum of the positive importances of the two noise columns")
    check(stable is True, "on three subsamples of the test set `ticket_code` always stays below 0.02", "n_repeats=5, random subsamples of the test set")
done()

## What you learned

- **A model that memorizes the training set makes any column "important":** on the training set the forest has AUC 1.0 and the permutation of `ticket_code` (a random integer) removes 0.02 of AUC, more than `age`; on the test set the same column is worth 0.013 and the ranking changes (and `random_number` drops to −0.003).
- **Any permutation importance must be computed on unseen data** (test set, validation, or on the OOB with implementations that support it): the training set contains the memory of the model, not its ability to generalize. It holds for the permutation, for the drop-column, for SHAP: compute them on validation data.
- With a "large" deep forest the phenomenon is stronger the more the model memorizes: it is another reason to use a higher `min_samples_leaf`, which makes the training set less perfect (and the importances on the training set less distorted).
- **A cheap check:** add two random columns (a continuous one, one with many integer values) to your dataset and check where they end up in the ranking. If a real feature is **below** the random columns, its importance cannot be distinguished from noise.

## Common pitfall

Computing the importance on the same set you trained on "because it is more convenient". With flexible models (forests, boosting) it is a method that rewards the memorized noise.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why must the permutation importance be computed on a validation set?</summary>

Because it measures how much the performance gets worse by shuffling a column: on the training set a flexible model has almost perfect performance and also uses noise columns to memorize, so their permutation makes it worse and they look important. On unseen data the performance reflects only what the model
learned that generalizes. A practical test is to add random columns and check that they come out at the bottom of the ranking.

</details>